In [ ]:
import os
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import torch
from torch import nn
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm


# ============================ НАСТРОЙКИ ============================
TEST_DIR = r'D:\kanopus_ikutsk\test_заболачивание' # ссылка на tif для формирования geoJson_pred
MODEL_PATH = 'best_model_segformer_заболачивание_main.pth'
MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"  # или b2
theshold = 0.9
PATCH_SIZE = 512
STRIDE = 256
BATCH_SIZE = 8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {DEVICE}')



# ============================ ЗАГРУЗКА МОДЕЛИ ============================
def load_model(model_name, model_path, in_channels=4, num_classes=2):
    # Загружаем предобученную модель
    model = SegformerForSemanticSegmentation.from_pretrained(
        model_name,
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
        use_safetensors=True
    )
    # Адаптируем первый свёрточный слой под in_channels
    first_conv = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d) and module.in_channels == 3:
            first_conv = module
            break
    if first_conv is None:
        raise ValueError("Не найден свёрточный слой с 3 входными каналами")
    new_conv = nn.Conv2d(
        in_channels=in_channels,
        out_channels=first_conv.out_channels,
        kernel_size=first_conv.kernel_size,
        stride=first_conv.stride,
        padding=first_conv.padding,
        bias=first_conv.bias is not None
    )
    with torch.no_grad():
        new_conv.weight[:, :3] = first_conv.weight
        new_conv.weight[:, 3] = first_conv.weight[:, 0]  # дублируем канал
    # Заменяем слой
    for name, module in model.named_modules():
        if module is first_conv:
            parent_name = name.rsplit('.', 1)[0] if '.' in name else ''
            parent = model.get_submodule(parent_name) if parent_name else model
            attr_name = name.rsplit('.', 1)[-1] if '.' in name else name
            setattr(parent, attr_name, new_conv)
            break
    model.config.num_channels = in_channels
    # Загружаем обученные веса
    state_dict = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    return model

model = load_model(MODEL_NAME, MODEL_PATH)
print('Модель загружена.')



def mask_to_geojson(mask, transform, crs, output_path, min_area=0):
    """
    Преобразует бинарную маску (0/1) в полигоны GeoJSON.
    Если в маске нет положительных пикселей, сохраняется пустой GeoJSON с той же схемой.
    """
    # Собираем геометрии в список (для обработки пустого случая)
    geometries = []
    for geom, value in shapes(mask, mask=(mask == 1), transform=transform):
        if value == 1:
            geometries.append({'geometry': geom, 'properties': {'class': int(value)}})

    if geometries:
        gdf = gpd.GeoDataFrame.from_features(geometries, crs=crs)
    else:
        # Создаём пустой GeoDataFrame с нужными колонками и CRS
        gdf = gpd.GeoDataFrame(columns=['class', 'geometry'], geometry='geometry', crs=crs)

    # Фильтрация по минимальной площади
    if min_area > 0 and not gdf.empty:
        gdf = gdf[gdf.geometry.area >= min_area]

    # Сохранение
    gdf.to_file(output_path, driver='GeoJSON')
    print(f"Сохранено {len(gdf)} полигонов в {output_path}")

def geojson_to_mask(geojson_path, transform, out_shape):
    """Растеризация GeoJSON в маску."""
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        return np.zeros(out_shape, dtype=np.uint8)
    shapes = [(geom, 1) for geom in gdf.geometry]
    mask = rasterize(shapes, out_shape=out_shape, transform=transform,
                     fill=0, dtype='uint8')
    return mask

def normalize_image(img):
    """Min-max нормализация по каждому каналу в [0, 1]."""
    img = img.astype(np.float32)
    for c in range(img.shape[0]):
        min_val = img[c].min()
        max_val = img[c].max()
        if max_val - min_val > 1e-6:
            img[c] = (img[c] - min_val) / (max_val - min_val)
        else:
            img[c] = 0
    return img
    
def sliding_window_inference(model, tif_path, patch_size=512, stride=256, batch_size=8):
    """Полное предсказание с взвешенным усреднением."""
    with rasterio.open(tif_path) as src:
        width, height = src.width, src.height

    prob_sum = np.zeros((height, width), dtype=np.float32)
    weight_sum = np.zeros((height, width), dtype=np.float32)

    # Гауссово окно для весов
    def create_weight_map(patch_size):
        ax = np.arange(patch_size)
        gauss = np.exp(-((ax - patch_size/2)**2) / (2*(patch_size/4)**2))
        weight = np.outer(gauss, gauss).astype(np.float32)
        return weight

    weight_map = create_weight_map(patch_size)

    # Список окон
    windows_list = []
    for y in range(0, height - patch_size + 1, stride):
        for x in range(0, width - patch_size + 1, stride):
            windows_list.append((x, y))

    for i in tqdm(range(0, len(windows_list), batch_size), desc='Inference'):
        batch_windows = windows_list[i:i+batch_size]
        batch_imgs = []
        batch_coords = []
        for x, y in batch_windows:
            with rasterio.open(tif_path) as src:
                window = Window(x, y, patch_size, patch_size)
                img = src.read(window=window)
            img = normalize_image(img)
            batch_imgs.append(img)
            batch_coords.append((x, y))

        imgs_tensor = torch.stack([torch.tensor(im, dtype=torch.float32) for im in batch_imgs]).to(DEVICE)
        with torch.no_grad():
            outputs = model(imgs_tensor)
            logits = outputs.logits
            logits = F.interpolate(logits, size=(patch_size, patch_size), mode='bilinear', align_corners=False)
            probs = torch.softmax(logits, dim=1)[:, 1]  # вероятность класса 1
            probs_np = probs.cpu().numpy()

        for (x, y), prob in zip(batch_coords, probs_np):
            prob_sum[y:y+patch_size, x:x+patch_size] += prob * weight_map
            weight_sum[y:y+patch_size, x:x+patch_size] += weight_map

    full_prob = prob_sum / np.maximum(weight_sum, 1e-6)
    pred_mask = (full_prob > theshold).astype(np.uint8)
    return pred_mask, full_prob

test_files = []
for fname in os.listdir(TEST_DIR):
    if fname.lower().endswith('.tif'):
        test_files.append( os.path.join(TEST_DIR, fname) )
print(f'Найдено тестовых сцен: {len(test_files)}')


for tif_path in test_files:
    print(f'Обработка: {os.path.basename(tif_path)}')
    # Читаем TIF для получения transform и размеров
    with rasterio.open(tif_path) as src:
        transform = src.transform
        crs = src.crs
        width, height = src.width, src.height

    # Предсказание
    pred_mask, _ = sliding_window_inference(model, tif_path, PATCH_SIZE, STRIDE, BATCH_SIZE)
    # Сохранение предсказанной маски как GeoJSON
    pred_geojson_path = os.path.join(TEST_DIR, f'pred_переувлажнение_{os.path.splitext(os.path.basename(tif_path))[0]}.geojson')
    mask_to_geojson(pred_mask, transform, crs, pred_geojson_path)
